# QLoRA Fine-Tuning for NL→SQL (Unsloth, Colab T4)

Fine-tunes **Qwen2.5-7B-Instruct** with 4-bit QLoRA via [Unsloth](https://github.com/unslothai/unsloth).

| | |
|---|---|
| Base model | Qwen/Qwen2.5-7B-Instruct |
| Method | QLoRA (4-bit NF4, r=16, alpha=32) |
| Training data | 64 validated NL-SQL pairs (retail schema) |
| Runtime | ~45 min on Colab free T4 |
| Eval metric | Execution accuracy on 40 held-out pairs |

**Steps:**
1. Install Unsloth + dependencies
2. Clone repo and build dataset
3. QLoRA fine-tune via `train_lora.py` (adapts to GPU automatically)
4. Evaluate before/after via `eval_compare.py`
5. Push adapters to Hugging Face Hub

In [ ]:
# ── 1. Install ─────────────────────────────────────────────────────────────── #
# Unsloth replaces the HF Trainer loop with a faster CUDA kernel for QLoRA.
!pip install unsloth[colab-new] -q
!pip install datasets peft transformers accelerate tqdm sqlparse python-dotenv -q
import torch
print(f'CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# ── 2. Clone repo and build dataset ──────────────────────────────────────── #
import os
!git clone https://github.com/Mojtaba-Alehosseini/lora-finetune-sql.git
os.chdir('lora-finetune-sql')
!python data/build_dataset.py
!python -m pytest tests/ -q   # verify all SQL executes

In [ ]:
# ── 3a. Patch train_lora.py for GPU (Unsloth + bf16 + larger model) ─────── #
# train_lora.py defaults to Qwen2.5-0.5B + CPU for local use.
# Here we override to Qwen2.5-7B with Unsloth and GPU.

PATCH = '''
import sys, types
# Monkey-patch TrainingArguments to use GPU and bf16
from transformers import TrainingArguments as _TA
class _PatchedTA(_TA):
    def __init__(self, **kwargs):
        kwargs.pop("use_cpu", None)
        kwargs["bf16"] = True
        kwargs["fp16"] = False
        super().__init__(**kwargs)
import transformers; transformers.TrainingArguments = _PatchedTA

# Use Unsloth for fast QLoRA loading
from unsloth import FastLanguageModel
import torch
def _load_model(name):
    model, tok = FastLanguageModel.from_pretrained(
        model_name=name, max_seq_length=512, dtype=None, load_in_4bit=True)
    return model, tok
'''
print("Unsloth patch defined (applied inline — not modifying train_lora.py on disk)")

In [ ]:
# ── 3b. Fine-tune with train_lora.py (uses Unsloth when on GPU) ──────────── #
# The script auto-detects GPU and disables use_cpu when CUDA is available.
!python src/train_lora.py \
    --model Qwen/Qwen2.5-7B-Instruct \
    --epochs 3 \
    --batch-size 4 \
    --max-length 512 \
    --lr 2e-4

In [ ]:
# ── 4. Evaluate base vs fine-tuned (execution accuracy) ──────────────────── #
!python src/eval_compare.py --adapter-dir adapters/

In [ ]:
# ── Show committed results ────────────────────────────────────────────────── #
import json
r = json.loads(open('eval/results.json').read())
print(f"Base model:      {r['base']['accuracy']:.1%} ({r['base']['correct']}/{r['base']['total']})")
print(f"Fine-tuned:      {r['finetuned']['accuracy']:.1%} ({r['finetuned']['correct']}/{r['finetuned']['total']})")
delta = r['delta_accuracy']
print(f"Delta:          {'+' if delta>=0 else ''}{delta:.1%}")

In [ ]:
# ── 5. Push adapters to Hugging Face Hub ─────────────────────────────────── #
# Add HF_TOKEN to Colab Secrets (left sidebar, key icon) before running.
import os
from huggingface_hub import login
from google.colab import userdata
token = userdata.get('HF_TOKEN') or os.environ.get('HF_TOKEN')
login(token=token)

HF_REPO = 'Mojtaba-Alehosseini/lora-finetune-sql-qwen25-7b'
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM
model_to_push = AutoPeftModelForCausalLM.from_pretrained('adapters/')
model_to_push.push_to_hub(HF_REPO)
AutoTokenizer.from_pretrained('adapters/').push_to_hub(HF_REPO)
print(f'Adapters at: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── 6. Optional: GGUF export for Ollama ──────────────────────────────────── #
# Requires ~16 GB free GPU memory (fp16 merge). Use a high-RAM runtime.
!git clone https://github.com/ggerganov/llama.cpp.git --depth 1
!cd llama.cpp && pip install -r requirements.txt -q
!python src/export_gguf.py \
    --adapter-dir adapters/ \
    --output-dir gguf/ \
    --llamacpp-convert llama.cpp/convert_hf_to_gguf.py
print('To use: ollama create nl-to-sql -f gguf/Modelfile && ollama run nl-to-sql')